# 🧠 DR Grading — RETFound Fine-tuning Pipeline

**Hướng dẫn sử dụng notebook này:**
1. Đảm bảo đã chọn **GPU runtime**: `Runtime → Change runtime type → T4 GPU`
2. Chạy từng cell **theo thứ tự từ trên xuống dưới**
3. Một số cell sẽ hỏi bạn nhập thông tin (token, xác nhận) — đọc kỹ hướng dẫn trong từng cell

---
**Thời gian ước tính:** ~30–60 phút setup, khoảng 2–4 ngày training (30 epochs với T4 GPU)

## ✅ Cell 1 — Kiểm Tra GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ KHÔNG TÌM THẤY GPU!\n"
        "Vào Runtime → Change runtime type → chọn T4 GPU → Save → chạy lại cell này."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"   CUDA version: {torch.version.cuda}")
print(f"   PyTorch version: {torch.__version__}")

## 📁 Cell 2 — Kết Nối Google Drive

Drive dùng để **lưu checkpoint** phòng khi Colab bị ngắt giữa chừng.
Một cửa sổ popup sẽ hiện ra → bấm **"Cho phép"** để cấp quyền.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/retfound_merged_seed42'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Checkpoint sẽ được lưu tại: {OUTPUT_DIR}")

## 📦 Cell 3 — Clone Source Code Từ GitHub

Repository được cấu hình sẵn từ tài khoản GitHub **Bang334**.

In [ ]:
# ============================================================
# ⚙️ CẤU HÌNH REPOSITORY
GITHUB_USERNAME = "Bang334"
GITHUB_REPO     = "dr-diagnostic-system" # Tên repo trên GitHub
GITHUB_BRANCH   = "feat/merged-dataset-training"  # Branch train bộ fundus gộp
# ============================================================

REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"
REPO_DIR = f"/content/{GITHUB_REPO}"

if os.path.exists(REPO_DIR):
    print("🔄 Repo đã tồn tại, đang chuyển đúng branch và pull bản mới nhất...")
    !git -C {REPO_DIR} fetch origin {GITHUB_BRANCH}
    !git -C {REPO_DIR} checkout {GITHUB_BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {GITHUB_BRANCH}
else:
    print(f"📥 Đang clone từ {REPO_URL}...")
    !git clone -b {GITHUB_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"✅ Working directory: {os.getcwd()}")
!ls -la

## 🔧 Cell 4 — Cài Đặt Dependencies

In [ ]:
print("📦 Đang cài thư viện cần thiết...")
!pip install -r ai/grading/requirements-train.txt -q
!pip install --upgrade "kaggle>=2.2.2" -q

# Kiểm tra các thư viện quan trọng
import timm, sklearn, cv2, PIL
from huggingface_hub import __version__ as hf_version
print(f"✅ timm: {timm.__version__}")
print(f"✅ huggingface_hub: {hf_version}")
print(f"✅ scikit-learn: {sklearn.__version__}")
print("✅ Tất cả thư viện đã sẵn sàng!")

## 🔑 Cell 5 — Đăng Nhập Hugging Face

RETFound-DINOv2 được host trên Hugging Face và cần xác thực.

**Trước khi chạy cell này, bạn phải:**
1. Tạo tài khoản tại [huggingface.co](https://huggingface.co)
2. Truy cập [huggingface.co/YukunZhou/RETFound_dinov2_meh](https://huggingface.co/YukunZhou/RETFound_dinov2_meh) → bấm **"Request access"** và chờ được duyệt
3. Vào [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → tạo token mới (loại **Read**)
4. Trong Colab, mở biểu tượng chìa khóa **Secrets**, tạo secret `HF_TOKEN` và bật quyền truy cập notebook

⚠️ Không ghi token trực tiếp vào notebook hoặc source code.

In [ ]:
import getpass
import subprocess
import sys
import zipfile
from pathlib import Path
import os
from google.colab import userdata
from huggingface_hub import login, whoami

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as exc:
    print(f"ℹ️ Không đọc được Colab Secret ({type(exc).__name__}); chuyển sang ô nhập ẩn.")
    hf_token = getpass.getpass("Dán Hugging Face Read token: ").strip()
if not hf_token:
    raise RuntimeError("HF_TOKEN chưa được cung cấp")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)
info = whoami(token=hf_token)
print(f"✅ Đã xác thực Hugging Face: {info['name']}")

## 🗂️ Cell 6 — Setup Kaggle và Tải Dataset Fundus Gộp

**Trước khi chạy, cấu hình Kaggle API Token trong Colab Secrets:**
1. Đăng nhập [kaggle.com](https://www.kaggle.com)
2. Click vào ảnh đại diện → **Settings**
3. Kéo xuống phần **API** → bấm **"Generate New Token"**
4. Trong Colab, mở biểu tượng chìa khóa **Secrets**, tạo secret tên `KAGGLE_API_TOKEN` và bật quyền truy cập notebook
5. Không ghi token trực tiếp vào notebook hoặc source code

Dataset: [Fundus (APTOS, DDR, IDRiD, EyePACS, Messidor)](https://www.kaggle.com/datasets/sehastrajits/fundus-aptosddridirdeyepacsmessidor) (~10.9 GB).

In [ ]:
import getpass
from google.colab import userdata
from ai.grading.train import find_predefined_splits

DATASET_DOWNLOAD_DIR = '/content/fundus_merged'
image_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp', '.webp'}

def resolve_dataset_dir():
    try:
        split_dirs = find_predefined_splits(Path(DATASET_DOWNLOAD_DIR))
    except (FileNotFoundError, ValueError):
        return None
    return next(iter(split_dirs.values())).parent

# Tự nhận train/val/test hoặc train/validation/test, kể cả khi có thư mục bao ngoài.
resolved_dataset_dir = resolve_dataset_dir()
if resolved_dataset_dir is not None:
    DATASET_DIR = str(resolved_dataset_dir)
    n_images = sum(
        1 for path in Path(DATASET_DIR).rglob('*')
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    print(f"✅ Dataset đã có sẵn: {n_images:,} ảnh trong {DATASET_DIR}")
else:
    try:
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        print(f"ℹ️ Không đọc được Kaggle Secret ({type(exc).__name__}); chuyển sang ô nhập ẩn.")
        kaggle_token = getpass.getpass("Dán Kaggle API token: ").strip()
    if not kaggle_token:
        raise RuntimeError("KAGGLE_API_TOKEN chưa được cung cấp")
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    print("✅ Kaggle API token đã được nạp vào runtime")

    Path(DATASET_DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)
    dataset_ref = "sehastrajits/fundus-aptosddridirdeyepacsmessidor"
    kaggle_cli = [sys.executable, "-m", "kaggle"]

    # Kiểm tra token và quyền truy cập trước khi tải file lớn.
    try:
        subprocess.run(
            kaggle_cli + ["datasets", "files", "-d", dataset_ref, "--page-size", "20"],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError as exc:
        detail = (exc.stderr or exc.stdout or str(exc)).strip()
        if "403" in detail or "Forbidden" in detail:
            raise RuntimeError("Kaggle trả về 403: hãy tạo lại API token có quyền Read.") from exc
        raise RuntimeError(f"Không kiểm tra được quyền Kaggle: {detail}") from exc

    print(f"\n⬇️ Đang tải bộ fundus gộp (~10.9 GB) vào {DATASET_DOWNLOAD_DIR}...")
    print("   Quá trình này thường mất khoảng 10–30 phút tuỳ tốc độ mạng.")
    subprocess.run(
        kaggle_cli + ["datasets", "download", "-d", dataset_ref, "-p", DATASET_DOWNLOAD_DIR],
        check=True,
    )

    print("\n📦 Đang giải nén...")
    zip_path = Path(DATASET_DOWNLOAD_DIR) / "fundus-aptosddridirdeyepacsmessidor.zip"
    if not zip_path.exists():
        raise FileNotFoundError(f"Kaggle báo thành công nhưng không tạo file: {zip_path}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(DATASET_DOWNLOAD_DIR)
    zip_path.unlink()

    resolved_dataset_dir = resolve_dataset_dir()
    if resolved_dataset_dir is None:
        discovered = sorted(
            path.relative_to(DATASET_DOWNLOAD_DIR).as_posix()
            for path in Path(DATASET_DOWNLOAD_DIR).rglob('*') if path.is_dir()
        )[:40]
        raise RuntimeError(
            "Giải nén xong nhưng không tìm thấy đủ train/val/test. "
            f"Các thư mục đầu tiên đã tìm thấy: {discovered}"
        )
    DATASET_DIR = str(resolved_dataset_dir)
    n_images = sum(
        1 for path in Path(DATASET_DIR).rglob('*')
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    print(f"\n✅ Hoàn tất! {n_images:,} ảnh đã sẵn sàng tại {DATASET_DIR}")

## 🔍 Cell 7 — Xem Trước Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

train_dir = Path(DATASET_DIR) / 'train'
records = [
    {'image_path': path, 'diagnosis': int(path.parent.name)}
    for path in train_dir.rglob('*')
    if path.is_file() and path.suffix.lower() in image_extensions
]
df = pd.DataFrame(records)
print("📊 Dataset info (train split):")
print(f"   Tổng số ảnh : {len(df):,}")
print(f"   Phân bố nhãn:\n{df['diagnosis'].value_counts().sort_index().to_string()}")
print()

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

# Vẽ biểu đồ phân bố và một ảnh mẫu cho mỗi lớp.
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
counts = df['diagnosis'].value_counts().sort_index()
axes[0].bar(CLASS_NAMES, counts.values, color=['#4CAF50','#8BC34A','#FF9800','#FF5722','#F44336'])
axes[0].set_title('Phân bố nhãn — merged fundus train')
axes[0].set_ylabel('Số lượng ảnh')
axes[0].tick_params(axis='x', rotation=25)

for grade in range(5):
    row = df[df['diagnosis'] == grade].sample(1, random_state=42).iloc[0]
    with Image.open(row['image_path']) as image:
        axes[grade + 1].imshow(image.convert('RGB'))
    axes[grade + 1].set_title(f"Grade {grade}: {CLASS_NAMES[grade]}")
    axes[grade + 1].axis('off')

plt.tight_layout()
plt.show()

## 🚀 Cell 8 — Bắt Đầu Training!

Đây là cell chính. Script sẽ:
- Giữ nguyên train/validation/test do tác giả dataset cung cấp
- Tải RETFound-DINOv2 từ HuggingFace
- Train 30 epochs với early stopping (patience=7)
- Lưu checkpoint tốt nhất vào Google Drive

**Thời gian ước tính: khoảng 2–4 ngày với T4 GPU (30 epochs)**

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-m", "ai.grading.train",
    "--dataset-dir", DATASET_DIR,
    "--output-dir",  OUTPUT_DIR,
    "--model-source", "retfound",
    "--retfound-id",  "RETFound_dinov2_meh",
    "--image-size",   "224",
    "--batch-size",   "2",
    "--accum-steps",  "8",
    "--freeze-epochs","3",
    "--epochs",       "30",
    "--loss",         "ce",
    "--balance",      "none",
    "--num-workers",  "2",
    "--seed",         "42",
]

print("🚀 Bắt đầu training...")
print("Command:", " ".join(cmd))
print("=" * 60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

if process.returncode == 0:
    print("\n✅ Training hoàn tất!")
else:
    print(f"\n❌ Training thất bại với mã lỗi: {process.returncode}")

## 🔄 Cell 8b — RESUME (Chạy Khi Colab Bị Ngắt)

Nếu Colab bị reset, chạy lại từ Cell 1 → 7, rồi chạy cell này thay vì Cell 8.

In [ ]:
LAST_CHECKPOINT = f"{OUTPUT_DIR}/checkpoint-last.pth"

if not os.path.exists(LAST_CHECKPOINT):
    print("❌ Không tìm thấy checkpoint để resume. Hãy chạy Cell 8 để bắt đầu từ đầu.")
else:
    import torch
    ckpt = torch.load(LAST_CHECKPOINT, map_location='cpu', weights_only=False)
    print(f"✅ Tìm thấy checkpoint tại epoch {ckpt['epoch']}, best QWK = {ckpt.get('best_qwk', 'N/A'):.4f}")
    print(f"🔄 Tiếp tục training từ epoch {ckpt['epoch'] + 1}...\n")

    cmd_resume = [
        sys.executable, "-m", "ai.grading.train",
        "--dataset-dir",  DATASET_DIR,
        "--output-dir",   OUTPUT_DIR,
        "--model-source", "retfound",
        "--retfound-id",  "RETFound_dinov2_meh",
        "--image-size",   "224",
        "--batch-size",   "2",
        "--accum-steps",  "8",
        "--freeze-epochs","3",
        "--epochs",       "30",
        "--loss",         "ce",
        "--balance",      "none",
        "--num-workers",  "2",
        "--seed",         "42",
        "--resume",       LAST_CHECKPOINT,
    ]

    process = subprocess.Popen(
        cmd_resume,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()

## 📊 Cell 9 — Xem Kết Quả Sau Training

In [ ]:
import json
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

# 1. Đọc lịch sử training
history_path = f"{OUTPUT_DIR}/history.jsonl"
if os.path.exists(history_path):
    records = [json.loads(l) for l in open(history_path)]
    epochs      = [r['epoch'] for r in records]
    train_losses= [r['train_loss'] for r in records]
    val_losses  = [r['val_loss'] for r in records]
    qwks        = [r['qwk'] for r in records]
    macro_f1s   = [r['macro_f1'] for r in records]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, train_losses, label='Train Loss', marker='o')
    axes[0].plot(epochs, val_losses,   label='Val Loss',   marker='o')
    axes[0].set_title('Training / Validation Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs, qwks,      label='QWK',      marker='o', color='green')
    axes[1].plot(epochs, macro_f1s, label='Macro F1', marker='o', color='orange')
    axes[1].set_title('Validation Metrics')
    axes[1].legend()
    axes[1].grid(True)
    plt.tight_layout()
    plt.show()

    best_epoch = max(records, key=lambda r: r['qwk'])
    print(f"\n🏆 Best epoch: {best_epoch['epoch']}")
    print(f"   QWK          : {best_epoch['qwk']:.4f}")
    print(f"   Macro F1     : {best_epoch['macro_f1']:.4f}")
    print(f"   Balanced Acc : {best_epoch['balanced_accuracy']:.4f}")

# 2. Xem test metrics cuối
test_metrics_path = f"{OUTPUT_DIR}/test_metrics.json"
if os.path.exists(test_metrics_path):
    print("\n📋 Test Metrics (trên tập test chưa từng thấy):")
    with open(test_metrics_path) as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# 3. Hiển thị confusion matrix
cm_path = f"{OUTPUT_DIR}/confusion_matrix_normalized.png"
if os.path.exists(cm_path):
    print("\n📊 Confusion Matrix:")
    display(IPyImage(cm_path))

## ⬇️ Cell 10 — Download Model Về Máy

In [ ]:
from google.colab import files as colab_files

best_ckpt = f"{OUTPUT_DIR}/checkpoint-best.pth"
test_csv  = f"{OUTPUT_DIR}/test_predictions.csv"
test_json = f"{OUTPUT_DIR}/test_metrics.json"

if os.path.exists(best_ckpt):
    size_mb = os.path.getsize(best_ckpt) / 1024**2
    print(f"📥 Đang tải checkpoint-best.pth ({size_mb:.0f} MB) về máy...")
    colab_files.download(best_ckpt)
else:
    print("❌ Chưa có checkpoint. Hãy chạy Cell 8 để train trước.")

# Tải thêm kết quả đánh giá (file nhỏ)
for path in [test_json, test_csv]:
    if os.path.exists(path):
        colab_files.download(path)
        print(f"✅ Đã tải: {os.path.basename(path)}")

print("\n💡 Lưu ý: Model PyTorch này dùng để tích hợp thay thế model_handler.py hiện tại.")